# Example usage

Example of how to use the merger_simulator package. The data is fictional. The specified demand model is a nested logit with two nests.

In [1]:
import pandas as pd
import merger_simulator as ms

df = pd.read_csv("fast_food.csv")

In [2]:
print(f"Shape: {df.shape}")
print(f"Markets: {df['market_id'].nunique()}, Cities: {df['city_id'].nunique()}, Quarters: {df['quarter_id'].nunique()}")
print(f"Firms: {df['firm'].value_counts().to_dict()}")
df.head()

Shape: (2651, 10)
Markets: 1000, Cities: 50, Quarters: 20
Firms: {'McDonalds': 1000, 'TacoBell': 1000, 'SushiHut': 651}


,firm,market_id,firm_id,city_id,quarter_id,market_size,cost_shifter,price,share,advertising
0,McDonalds,1,1,1,1,4608,0.638277,2.481150,0.013190,1
1,TacoBell,1,2,1,1,4608,-0.466893,2.150774,0.014490,0
2,SushiHut,1,3,1,1,4608,-0.574624,2.930918,0.113606,1
3,McDonalds,2,1,1,2,1541,1.191405,2.878074,0.082200,1
4,TacoBell,2,2,1,2,1541,0.468009,2.411335,0.012450,1


Each city-quarter is considered as one unique market. McDonald's and TacoBell are present in all markets. SushiHut, the entrant, is present only in some markets.

In [3]:
# Prep columns (firm dummies for fixed effects)
df["const"] = 1
df["is_tacobell"] = (df["firm"] == "TacoBell").astype(int)
df["is_sushihut"] = (df["firm"] == "SushiHut").astype(int)

In [4]:
# 1. Wrap data
data = ms.MergerData(
    df,
    firm_col="firm",
    market_col="market_id",
    price_col="price",
    share_col="share",
    market_size_col="market_size",
    cost_shifter_cols=["cost_shifter"],
    product_chars=["advertising"],
    nest_col="nest",
    firm_id_col="firm_id",
)

We use a **nested logit** demand model. The plain logit imposes IIA, meaning diversion ratios are proportional to market shares. This is unrealistic for merger analysis because when a consumer leaves McDonalds, they are more likely to switch to another fast food restaurant than to the outside good. The nested logit relaxes IIA by allowing correlated preferences within a group.

For the nest structure, we decided to have *two* nests: one nest for McDonalds and TacoBell, and one nest for SushiHut. McDonalds and Taco Bell are both traditional American fast food chains with similar price points, similar service model, and similar target demographics. Sushi Hut, however, is a new entrant that is a fundamentally different cuisine category (tuna and other fish products compared to burgers and "TexMex" food). So, we believe that a consumer choosing between a burger and a taco is making a different kind of decision than choosing between a taco and sushi. Nests should group products that are closer substitutes due to unobserved characteristics.

The Berry inversion gives:

$$\ln(s_{jt}) - \ln(s_{0t}) = \beta_0 + \beta_1 \text{advertising}_{jt} + \text{firm FEs} - \alpha \cdot p_{jt} + \sigma \cdot \ln(s_{j|g,t}) + \xi_{jt}$$

where $s_{j|g,t} = s_{jt} / \sum_k s_{kt}$ is product $j$'s share within its nest and $\sigma = (1 - \lambda) \in [0,1)$ is the nesting parameter. Note that for SushiHut, $s_{SH|g} = s_{SH} / s_{SH} = 1$ always, so $\ln(s_{SH|g}) = 0$. This means that the $\sigma$ parameter governs within-nest substitution for the McD/TB nest only, and a singleton nest doesn't need its own nesting parameter because there is no within-nest substitution to model. Also note that in our specification we have $-\alpha$. The regression will find a negative alpha (due to law of demand), but we flip the sign to avoid sign confusion during elasticities calculations downstream (so $\alpha$ would be positive), so we flip back the sign so that the coefficient on price in the model is negative again.

In [ ]:
# 2. Choose demand model
model = ms.NestedLogit(nests={"legacy": ["McDonalds", "TacoBell"], "new": ["SushiHut"]})
model.apply_nests(data)

**Instruments:** Both price and $\ln(s_{j|g})$ are endogenous. We instrument using:

1. **Cost shifters:** Firm-specific cost shifters shift prices but should not directly affect demand. We know cost shifters are valid instruments for price because they raise input costs but do not directly affect demand.
2. **Rival cost shifters:** Competitors' cost shifters affect own price through competitive interactions. When rival's costs rise, their prices rise, which affect their demand within the nest, changing within-nest shares for all nest members (relevance for $\ln(s_{j|g})$). But, rival input costs don't directly enter demand directly (e.g. a consumer doesn't care whether TB pays more for corn when deciding whether to buy from TB or McD).
3. **BLP instruments:** Number of firms in the market; average rivals' advertising. The number of firms is relevant for price because more competition lowers equilibrium prices. For exclusion, we know that the number of firms is determined by entry decisions on market-level profitability, not by idiosyncratic demand shocks for individual products. In class, we saw how this potential instrument could violate the exclusion condition if firms enter because they observe demand, but we assumed the exclusion condition holds. Rival's advertising is also relevant because it affects the rival's prices (as advertising shifts demand), so it shifts within-nest shares. Rival's advertising also meets the exclusion condition because a rival's adversiting should not directly affect your own demand conditional on your own adverstising, price, and product characteristics.
4. **Hausman instruments** (if no cost shifter): Prices of the same firm in other markets. We have seen in class how Hausman instruments meet both conditions: they are relevant because a firm facing common cost shocks will have correlated prices across markets, and meets the exclusion condition because demand shocks in one market should be independent of demand shocks in another market (controlling for time).

In [6]:
# 3. Build instruments (within_nest_rival_cost derives nests from the model)
iv = (
    ms.hausman(data, time_col="quarter_id")
    + ms.rival_cost_shifters(data)
    + ms.blp_instruments(data)
    + ms.within_nest_rival_cost(data)
)

In [7]:
# 4. Estimate demand
result = model.estimate(
    data, instruments=iv,
    endog_cols=["price", "_ln_within_nest_share"],
    exog_cols=["const", "advertising", "is_tacobell", "is_sushihut"],
)
print(result.summary())
print(f"alpha = {model.alpha:.4f}, sigma = {model.sigma:.4f}")

                       estimate  std_error     t_stat
parameter                                            
const                  1.631429   0.270778   6.024967
advertising            0.458021   0.040393  11.339094
is_tacobell            0.000258   0.033268   0.007760
is_sushihut           -0.506094   0.060553  -8.357915
price                 -2.009595   0.127172 -15.802142
_ln_within_nest_share  0.776004   0.036238  21.414158
alpha = 2.0096, sigma = 0.7760


In the nested logit, marginal cost is recovered from the pricing first-order condition: $$mc_j = p_j - \underbrace{\left[-(\Omega \odot \Delta)^{-1} \mathbf{s}\right]_j}_{\text{markup}}$$ Where:
  - $\Omega$ is the ownership matrix (1 if firms $i,j$ are co-owned, 0 o/w)
  - $\odot$ is element-wise multiplication
  - $\mathbf{s}$ is the vector of shares
  - $\Delta$ is the matrix of share derivatives with entries: $$\Delta_{jj} = \frac{\partial s_j}{\partial p_j} = -\alpha \cdot s_j \left[\frac{1}{1-\sigma} - \frac{\sigma}{1-\sigma} s_{j|g} - s_j \right]  \quad \text{(own-price)}$$ 
  $$\Delta_{jk} = \frac{\partial s_j}{\partial p_k} = \alpha \cdot s_j \left[\frac{\sigma}{1-\sigma} s_{k|g} + s_k \right] \quad \text{(same nest)}$$
  $$\Delta_{jk} = \frac{\partial s_j}{\partial p_k} = \alpha \cdot s_j \cdot s_k \quad \text{(different nests)}$$
  
This comes from the Bertrand-Nash FOC, where firms maximize profits. 
 
The key difference from logit is that within-nest cross-derivatives have an extra $\frac{\sigma}{1-\sigma} s_{k|g}$ term, making McDonalds and Taco Bell closer substitutes. This makes more sense than assuming McDonalds, TacoBell, SushiHut, and the outside good are equally substitutable. This means the merger will produce larger price increases. 

The $\text{merger\_simulator}$ package handles this computation.

In [8]:
# 5. Recover marginal costs
ms.recover_marginal_costs(model, data)
print(data.df.groupby("firm")[["_mc", "_markup"]].mean())

                _mc   _markup
firm                         
McDonalds  2.123194  0.231333
SushiHut   1.813915  0.526257
TacoBell   2.155275  0.221560


McD and TB have similar cost structures ($\approx \$2.12–2.15$) with thin margins ($\approx 10\%$), while SushiHut has much lower costs ($\approx \$1.81$) but charges a much higher markup ($\approx \$0.53$, or $\approx 22\%$ margin). This makes economic sense because McD/TB compete intensely within their nest, driving markups down to about $\$0.22$, while SushiHut faces no within-nest rival and can exercise more market power despite its lower baseline appeal. The overall mean margin of $\approx 13\%$ is reasonable for fast food. The fact that McD and TB have nearly identical markups (consistent with their estimated identical demand-side fixed effects) confirms that symmetric firms in the same nest should behave symmetrically.

In [9]:
# 6. Simulate merger
merger = ms.simulate_merger(model, data, merging_firms=("McDonalds", "TacoBell"))
print(merger.summary())

           mean_pct_change  median_pct_change  min_pct_change  max_pct_change
firm                                                                         
McDonalds          12.5515            13.8747          0.0087         28.0811
SushiHut            0.0203             0.0128          0.0000          0.1842
TacoBell           12.9307            14.0637          0.1011         24.2245


The merger simulation shows McD and TB raise prices by about 12.6–12.9\% on average, while SushiHut is essentially unaffected (~0.02\%). This is exactly what the nested logit predicts: since McD/TB are in the same nest with $\sigma = 0.78$, the merger internalizes the strong within-nest substitution that previously constrained pricing. Pre-merger, if McD raised prices, it lost customers mostly to TB, but now they're the same firm, so that diversion is profit-neutral, and both raise prices substantially. SushiHut barely benefits because cross-nest diversion is governed only by the small $s_j \cdot s_k$ term. The wide range of price changes (0\% to 28\% for McD) reflects market heterogeneity: in 2-firm markets the merger creates a monopoly, while in 3-firm markets SushiHut's presence provides at least some competitive pressure.

In [10]:
# 7. Welfare
welfare = ms.compute_welfare(merger, alpha=model.alpha, merging_firms=("McDonalds", "TacoBell"))
print(welfare)

=== Welfare Summary ===
Consumer surplus change:  -13.5653
Producer surplus change:  +141820.3173
Total surplus change:     +141806.7520
HHI pre:  5527
HHI post: 7498
HHI change: +1972
Price index change: +7.0580%


We see that the proposed merger between McDonald's and TacoBell would raise the presumption of illegality under the 2023 Merger Guidelines because the pre-merger HHI is $>1800$ (highly concentrated market) and the post-merger change in HHI is $>100$. Moreover, the proposed merger raises overall prices, allowing the merged party to raise markups and extract more profits from consumers. Thus, the merged party would have the incentives to raise prices. Also, since in this hypothetical McDonald's and TacoBell have no other within-nest competitors, they have the ability to raise prices without losing too many consumers to competitors or the outside good.